# Hari 1, Sesi 1: Regresi Linear, dari Nol sampai Bisa Dibaca

**Durasi:** 75 menit

Regresi linear adalah alat utama di ekonometrika. Konsepnya sederhana: cari
satu garis yang paling dekat dengan semua titik data, dengan "dekat" diukur
sebagai jumlah kuadrat selisih vertikal (residual).

$$Y_i = \beta_0 + \beta_1 X_{1i} + \beta_2 X_{2i} + \epsilon_i$$

Materi sesi ini:
1. Regresi sederhana dan cara membaca output statsmodels.
2. Regresi berganda dan arti koefisien secara ekonomi.
3. Uji t, uji F, R kuadrat, dan R kuadrat terkoreksi (adjusted R2).
4. Prediksi dengan model.

Setiap bagian punya catatan "cara menjelaskan ke mahasiswa" supaya sesi ini
bisa langsung dipakai mengajar.



**Posisi dalam alur workshop.** Dari alat (Sesi 0) ke model pertama
(Sesi 1). Di bagian 7 sesi ini model linear sengaja ditunjukkan gagal
pada hubungan melengkung; Sesi 2 memperbaikinya.



## Persiapan: pastikan data tersedia

Notebook ini membaca CSV dari folder `data/`. Kalau belum ada (misalnya di
Colab), unggah `data-ekonometrika.zip` lewat panel Files, lalu jalankan
sel di bawah sekali. Di komputer sendiri, ekstrak zip di folder yang sama.



In [ ]:
import zipfile
from pathlib import Path

if not Path("data").exists() and Path("data-ekonometrika.zip").exists():
    zipfile.ZipFile("data-ekonometrika.zip").extractall(".")

jumlah = len(list(Path("data").glob("*.csv")))
print(f"{jumlah} file data siap dipakai dari folder data/")
if jumlah == 0:
    print("Folder data/ kosong. Unggah data-ekonometrika.zip dulu, lalu jalankan sel ini lagi.")



## 1. Data dan pertanyaan penelitian

Kasus dari `data/uts-2.csv`: apa yang menjelaskan harga saham 50
perusahaan? Variabel: pendapatan per saham (EPS), rasio utang terhadap
ekuitas (DER), dan belanja iklan.

Kita mulai dari satu variabel dulu: EPS. Pertanyaan sederhananya,
"perusahaan dengan EPS lebih tinggi, apakah harga sahamnya juga lebih
tinggi?"



In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette=["#2F5D50", "#C15937", "#D9A441", "#5B8A8F"])
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "font.size": 11})

saham = pd.read_csv("data/uts-2.csv").rename(columns={
    "Y_Harga_Saham": "harga",
    "X1_Pendapatan_per_Saham": "eps",
    "X2_Rasio_Utang_Ekuitas": "der",
    "X3_Belanja_Iklan": "iklan",
})
saham.head()



## 2. Lihat data dulu, selalu

Satu scatter plot sebelum regresi. Polanya hampir linear sempurna, sinyal
bahwa regresi linear masuk akal untuk hubungan ini.



In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(saham["eps"], saham["harga"], color="#2F5D50", s=30)
ax.set_xlabel("Pendapatan per saham (EPS)")
ax.set_ylabel("Harga saham")
ax.set_title("EPS vs harga saham")
plt.show()



## 3. Regresi sederhana

Rumus di statsmodels ditulis `y ~ x`. Intersep ditambahkan otomatis.



In [ ]:
model1 = smf.ols("harga ~ eps", data=saham).fit()
print(model1.summary())



### Cara membaca tabel output

Tiga angka yang paling sering dipakai:

- `coef` pada baris `Intercept`: nilai Y saat X = 0. Di kasus ini artinya
  harga saham teoretis saat EPS nol. Tidak selalu masuk akal secara bisnis,
  tapi tetap harus dilaporkan.
- `coef` pada baris `eps`: perubahan Y untuk tambahan satu unit X.
  Inilah jawaban dari pertanyaan penelitian.
- `P>|t|`: peluang mengamati koefisien sebesar ini kalau pengaruh
  sebenarnya nol. Di bawah 0,05 artinya signifikan pada taraf 5%.

Kolom `R-squared` mengukur proporsi variasi Y yang diterangkan model.
Hati-hati: data simulasi seperti `uts-2.csv` punya R kuadrat sangat tinggi
karena dibangkitkan dari formula yang rapi. Di data riil, R kuadrat 0,3
sampai 0,6 untuk data cross-section sudah wajar.



In [ ]:
b0 = model1.params["Intercept"]
b1 = model1.params["eps"]
print(f"Intersep  : {b0:.3f}")
print(f"Slope     : {b1:.3f}")
print(f"R-squared : {model1.rsquared:.3f}")
print(f"p-value eps: {model1.pvalues['eps']:.3g}")



### Visual: garis OLS dan residual

Gambar ini memperlihatkan cara kerja OLS: garis merah adalah hasil
estimasi, garis abu-abu adalah residual yang dikuadratkan lalu
diminimalkan jumlahnya.



In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(saham["eps"], saham["harga"], color="#2F5D50", s=28, zorder=3)
xs = np.linspace(saham["eps"].min(), saham["eps"].max(), 50)
ax.plot(xs, b0 + b1 * xs, color="#C15937", lw=2, zorder=2)
# Tarik garis residual untuk 12 titik sampel saja supaya gambar terbaca
for _, baris in saham.sample(12, random_state=1).iterrows():
    y_hat = b0 + b1 * baris["eps"]
    ax.plot([baris["eps"]] * 2, [baris["harga"], y_hat],
            color="gray", lw=0.9, zorder=1)
ax.set_xlabel("Pendapatan per saham (EPS)")
ax.set_ylabel("Harga saham")
ax.set_title(f"Harga = {b0:.1f} + {b1:.1f} x EPS")
plt.show()




### Coba sendiri (5 menit)

Ganti variabel penjelas dari `eps` ke `der` dan jalankan regresinya.
Apakah `der` signifikan? Lalu, kenapa hasilnya berbeda dengan model
berganda di bagian 6 nanti?

Tulis di sel di bawah, baru buka jawabannya.



In [ ]:
# Tulis kode Anda di sini



<details><summary>Lihat jawaban</summary>

Sendirian, `der` tampak sangat signifikan (koefisien -131,3; p = 0,0000;
R2 = 0,93). Tapi di model berganda (bagian 6) `der` justru tidak
signifikan. Pelajaran: koefisien sederhana bisa menyesatkan karena
mencampur pengaruh variabel lain, dalam hal ini EPS.

</details>



## 4. Arti koefisien dengan angka

Dari output di atas, slope sekitar 10,4. Artinya: kenaikan EPS satu unit
diiringi kenaikan harga saham sekitar 10,4 unit, dengan asumsi hal lain
tidak berubah. Intersep sekitar 87,8 adalah titik potong teoretis di EPS = 0.



## 5. Prediksi

Prediksi adalah alasan orang membangun model. Di statsmodels, pakai
`predict` dengan DataFrame berisi nilai X baru. Perhatikan bentuk datanya:
harus DataFrame, bukan angka tunggal.



In [ ]:
baru = pd.DataFrame({"eps": [15.0, 20.0, 25.0]})
baru["prediksi_harga"] = model1.predict(baru)
baru



Prediksi manual untuk satu nilai juga mudah: `b0 + b1 * eps`. Hasilnya
harus sama persis dengan `model1.predict(pd.DataFrame({"eps": [15]}))`,
selisih hanya mungkin dari pembulatan.



## 6. Regresi berganda

Sekarang kasusnya dilengkapi: harga saham dijelaskan EPS, DER, dan belanja
iklan sekaligus.



In [ ]:
model2 = smf.ols("harga ~ eps + der + iklan", data=saham).fit()
print(model2.summary())



### Uji F: apakah model berguna secara keseluruhan?

`Prob (F-statistic)` menguji hipotesis bahwa semua slope sama dengan nol.
Kalau p-value kecil, setidaknya ada satu variabel yang menjelaskan Y.



In [ ]:
print(f"F-statistic : {model2.fvalue:.3f}")
print(f"Prob (F)    : {model2.f_pvalue:.6g}")
print(f"R2          : {model2.rsquared:.4f}")
print(f"Adj R2      : {model2.rsquared_adj:.4f}")



### Koefisien satu per satu

Uji t di tabel output adalah uji parsial: apakah satu koefisien signifikan
setelah variabel lain diperhitungkan. Ini berbeda dari korelasi biasa,
karena pengaruh variabel lain sudah "dibersihkan".



In [ ]:
tabel = pd.DataFrame({
    "koefisien": model2.params,
    "std_err": model2.bse,
    "t": model2.tvalues,
    "p_value": model2.pvalues,
})
tabel



Di data ini, EPS dan iklan signifikan, sementara DER tidak. Kenaikan DER
satu unit berhubungan dengan kenaikan harga 2,94 unit, tapi p-value 0,48
artinya angka itu belum bisa dibedakan dari nol secara statistik.




### Coba sendiri (5 menit)

Pakai `model2` (harga ~ eps + der + iklan). Prediksi harga untuk
EPS = 18, DER = 1,5, dan belanja iklan = 60.

Tulis di sel di bawah, baru buka jawabannya.



In [ ]:
# Tulis kode Anda di sini



<details><summary>Lihat jawaban</summary>

`model2.predict(pd.DataFrame({"eps": [18], "der": [1.5], "iklan": [60]}))`
menghasilkan sekitar 275,6.

</details>



## 7. Kapan garis lurus tidak cukup

Coba regresikan penjualan pada biaya iklan memakai `regresi-polinom.csv`.
Hubungannya melengkung, dan model linear gagal total: R kuadrat hanya
sekitar 0,007 dan slope tidak signifikan. Ini bukan alasan untuk menyerah,
tapi tanda bahwa bentuk fungsionalnya perlu diganti. Itu topik Sesi 2.



In [ ]:
iklan = pd.read_csv("data/regresi-polinom.csv").rename(columns={
    "Biaya Iklan (X)": "iklan", "Penjualan (Y)": "penjualan"})
model_salah = smf.ols("penjualan ~ iklan", data=iklan).fit()
print(f"R2 linear  : {model_salah.rsquared:.4f}")
print(f"p-value iklan: {model_salah.pvalues['iklan']:.3f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(iklan["iklan"], iklan["penjualan"], color="#2F5D50", s=26)
ax.plot(iklan["iklan"], model_salah.fittedvalues, color="#C15937", lw=1.5)
ax.set_xlabel("Biaya iklan")
ax.set_ylabel("Penjualan")
ax.set_title("Model linear pada hubungan yang melengkung")
plt.show()



## 8. Catatan mengajar: tiga kesalahan yang paling sering muncul

**Satu, menafsirkan intersep tanpa berpikir.** "Harga saham 87,8 padahal
EPS nol" sering diucapkan seolah itu temuan. Intersep hanya berguna kalau
X = 0 masuk akal secara ekonomi.

**Dua, menyamakan R kuadrat tinggi dengan model benar.** Model dengan R
kuadrat 0,95 bisa saja salah spesifikasi, misalnya karena tren palsu
(spurious regression) pada data deret waktu. Ingat sesi 5.

**Tiga, lupa satuan.** Koefisien 10,4 pada EPS diukur dalam satuan Y per
satuan X. Kalau Y rupiah dan X persen, 10,4 berarti 10,4 rupiah per satu
persen, bukan per 0,01 persen.

Satu lagi yang sering luput: koefisien bersifat *ceteris paribus*, bukan
pengaruh kausal. Bilang "berhubungan dengan" kecuali desain penelitiannya
memang memungkinkan klaim kausal.



## Latihan

1. Regresikan `produktivitas padi` pada pupuk, curah hujan, dan sinar
   matahari (`uts-3.csv`). Laporkan koefisien, p-value, dan R kuadrat.
2. Prediksi produktivitas kalau pupuk 160, curah hujan 1900, sinar 230.
3. Setelah model Latihan 1 jadi (variabel `model_padi`), uji apakah
   pupuk dan sinar matahari bersama-sama signifikan:
   `model_padi.f_test("X1_Jumlah_Pupuk = 0, X3_Sinar_Matahari = 0")`.

Kunci dan interpretasi ada di `kunci_latihan.md`.



### Selanjutnya

Selanjutnya: Sesi 2 menambal celah yang terlihat di bagian 7, yaitu
model linear pada hubungan melengkung.
